# FF BB defense by rank

Reproducibility check for the lesson artifact: validate the 60 aggregate slices, print the default BTN / 2 BB ladder, and recompute the ecological Pearson correlation between log ABI and defend frequency. This is a four-cohort association, not a causal estimate.

In [ ]:
from pathlib import Path
import json
import math

DATA_PATH = Path("../data/ff-bb-defense-ranks.json")
payload = json.loads(DATA_PATH.read_text(encoding="utf-8"))

COHORTS = ("novice", "league3", "league2", "league1")
POSITIONS = ("EP", "MP", "HJ", "CO", "BTN")
SIZES = ("2_0", "2_5", "3_0")
expected_keys = {f"{cohort}:{position}:{size}" for cohort in COHORTS for position in POSITIONS for size in SIZES}
aggregates = payload["aggregates"]
assert len(aggregates) == 60, f"expected 60 aggregate cells, got {len(aggregates)}"
assert set(aggregates) == expected_keys, "aggregate cube is incomplete or contains unexpected keys"

for key, row in aggregates.items():
    for field in ("n", "players", "folds", "calls", "threeBets", "cardKnownN"):
        value = row[field]
        assert isinstance(value, int) and not isinstance(value, bool) and value >= 0, f"{key}.{field}"
    assert row["folds"] + row["calls"] + row["threeBets"] == row["n"], f"{key}: action sum"
    assert row["players"] <= row["n"], f"{key}: players exceed decisions"
    assert row["cardKnownN"] <= row["n"], f"{key}: known-card count exceeds decisions"

abi = {cohort: float(payload["meta"]["abi"][cohort]["abiUsd"]) for cohort in COHORTS}
defend = {}
print("Default spot: BB vs BTN open 2 BB")
for cohort in COHORTS:
    row = aggregates[f"{cohort}:BTN:2_0"]
    defend[cohort] = 100.0 * (row["calls"] + row["threeBets"]) / row["n"]
    three_bet = 100.0 * row["threeBets"] / row["n"]
    label = payload["meta"]["cohorts"][cohort]["label"]
    print(f"{label:>16}: ABI ${abi[cohort]:>5.2f} | defend {defend[cohort]:>5.1f}% | 3-bet {three_bet:>5.1f}% | n={row['n']:,}")

def pearson(xs, ys):
    x_mean = sum(xs) / len(xs)
    y_mean = sum(ys) / len(ys)
    numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
    denominator = math.sqrt(sum((x - x_mean) ** 2 for x in xs) * sum((y - y_mean) ** 2 for y in ys))
    return numerator / denominator

r = pearson([math.log(abi[cohort]) for cohort in COHORTS], [defend[cohort] for cohort in COHORTS])
stored_r = float(payload["meta"]["abiCorrelation"]["pearsonR"])
print(f"Pearson r(log ABI, defend), four cohort aggregates: {r:.3f}")
assert math.isclose(r, stored_r, abs_tol=0.002), f"stored r={stored_r:.3f}, recomputed r={r:.3f}"
print("Validation: OK")